<a href="https://colab.research.google.com/github/salty-arch/Flyrank-Intern/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [58]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.sql(f"CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

rel = "hf://datasets/FlyRank/internship-warehouse"

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I'm using **Logistic Regression** and **Random Forest**, since this is a binary
classification problem (`declining_flag`: True/False).

I chose Logistic Regression first because I recently studied it in another ML course
(Andrew Ng's Machine Learning Specialization) and understand its mechanics the cost
function, decision boundaries, and how it's optimized via gradient descent. It's also
easier to interpret than more complex models, and interpretability matters here
specifically: my Week 2 framing described this score as a **decision-support tool**
a human reviewer acts on it, the model doesn't decide anything on its own. Being able
to explain *why* a page was flagged (which features pushed the prediction) is valuable
for that reviewer, not just a nice-to-have.

One method alone didn't feel like enough evidence, so I'm also training a **Random
Forest**. It trains many decision trees independently (in parallel) and combines their
votes, which makes it more robust than a single model and better able to capture
complex, non-linear patterns that Logistic Regression might miss.

Comparing both models against each other, and both against my Week 4 baseline, gives
a fuller picture of whether ML actually earns its place here not just whether one
particular model happens to work.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

The data will be grouped by client  because if the same client ends up in both training and testing, the model might learn that specific clients pattern and cheat its way to the right answer, by keeping a client entirely in training set or testing set the model will be forced to learn the pattern for the problem itself and not the client.

I'm using GroupShuffleSplit, grouped by client_hash_id, with an 80/20 train/test split. I verified this split is honest by checking for client overlap between the two sets, zero clients appear in both, confirming the model will be tested only on clients it never saw during training.

In [59]:
feature_frame = con.sql(f"""
    WITH daily AS (
        SELECT
            content_hash_id,
            client_hash_id,
            report_date,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position,
            CASE WHEN report_date < DATE '2026-03-16' THEN 'first_half' ELSE 'second_half' END AS period
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
        WHERE gsc_data_available IS TRUE
    ),
    features AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(gsc_impressions) AS feat_impressions,
            SUM(gsc_clicks) AS feat_clicks,
            AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS feat_avg_position,
            COUNT(*) AS feat_days_active,
            SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS feat_ctr
        FROM daily
        WHERE period = 'first_half'
        GROUP BY content_hash_id, client_hash_id
    ),
    label AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(CASE WHEN period = 'first_half' THEN gsc_impressions ELSE 0 END) AS first_half_impressions,
            SUM(CASE WHEN period = 'second_half' THEN gsc_impressions ELSE 0 END) AS second_half_impressions
        FROM daily
        GROUP BY content_hash_id, client_hash_id
    )
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        f.feat_impressions,
        f.feat_clicks,
        f.feat_avg_position,
        f.feat_days_active,
        f.feat_ctr,
        CASE
            WHEN (l.second_half_impressions - l.first_half_impressions) * 1.0 / NULLIF(l.first_half_impressions, 0) * 100 <= -20
            THEN TRUE ELSE FALSE
        END AS declining_flag
    FROM features f
    JOIN label l
        ON f.content_hash_id = l.content_hash_id AND f.client_hash_id = l.client_hash_id
    WHERE l.first_half_impressions > 0
""").df()

print(feature_frame.shape)
print(feature_frame.head())
print(feature_frame["declining_flag"].value_counts())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(151981, 8)
            content_hash_id           client_hash_id  feat_impressions  \
0  content_05597932fe4da067  client_73cda7b4e4f265ea              18.0   
1  content_7a105f548d9c6916  client_73cda7b4e4f265ea            4173.0   
2  content_905aa32a0230694e  client_73cda7b4e4f265ea              89.0   
3  content_a3ea9792f793ec72  client_73cda7b4e4f265ea             245.0   
4  content_36c36abc7650d7af  client_73cda7b4e4f265ea            3705.0   

   feat_clicks  feat_avg_position  feat_days_active  feat_ctr  declining_flag  
0          0.0           9.055556                11  0.000000           False  
1          6.0           6.327311                15  0.001438            True  
2          0.0           3.763426                15  0.000000            True  
3          0.0           4.185913                15  0.000000           False  
4          3.0           6.473735                15  0.000810            True  
declining_flag
False    101981
True      50000
Name: count, dty

In [60]:
def assign_tier(pos):
    if pos <= 3:
        return "top_3"
    elif pos <= 10:
        return "page_1"
    elif pos <= 20:
        return "striking"
    elif pos <= 50:
        return "page_3_5"
    else:
        return "deep"

feature_frame["position_tier"] = feature_frame["feat_avg_position"].apply(assign_tier)

tier_medians = feature_frame[feature_frame["feat_ctr"] > 0].groupby("position_tier")["feat_ctr"].median()
print(tier_medians)

position_tier
deep        0.011174
page_1      0.003891
page_3_5    0.002268
striking    0.004673
top_3       0.004124
Name: feat_ctr, dtype: float64


In [61]:
feature_frame["tier_median_ctr"] = feature_frame["position_tier"].map(tier_medians)

In [62]:
feature_frame["ctr_mismatch"] = (
    (feature_frame["feat_ctr"] < (feature_frame["tier_median_ctr"] * 0.5))
    & (feature_frame["feat_impressions"] >= 100)
)

print(feature_frame["ctr_mismatch"].value_counts())

ctr_mismatch
False    108502
True      43479
Name: count, dtype: int64


In [63]:
feature_frame["combined_flag"] = feature_frame["declining_flag"] & feature_frame["ctr_mismatch"]

print(feature_frame["combined_flag"].value_counts())

combined_flag
False    137299
True      14682
Name: count, dtype: int64


In [64]:
from sklearn.model_selection import GroupShuffleSplit

splitter2 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_idx2, test_idx2 = next(splitter2.split(
    feature_frame,
    groups=feature_frame["client_hash_id"]
))

train_df2 = feature_frame.iloc[train_idx2].dropna(subset=["feat_avg_position"])
test_df2 = feature_frame.iloc[test_idx2].dropna(subset=["feat_avg_position"])

train_clients2 = set(train_df2["client_hash_id"])
test_clients2 = set(test_df2["client_hash_id"])
print(f"Clients in both train and test: {len(train_clients2 & test_clients2)}")

print("Train:", train_df2.shape, "Test:", test_df2.shape)

Clients in both train and test: 0
Train: (137554, 12) Test: (13121, 12)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [65]:
features_list = ["feat_impressions", "feat_clicks", "feat_avg_position", "feat_days_active", "feat_ctr"]

X_train2 = train_df2[features_list]
y_train2 = train_df2["combined_flag"]
X_test2 = test_df2[features_list]
y_test2 = test_df2["combined_flag"]

print("X_train2:", X_train2.shape, "X_test2:", X_test2.shape)
print(y_train2.value_counts())

X_train2: (137554, 5) X_test2: (13121, 5)
combined_flag
False    123909
True      13645
Name: count, dtype: int64


In [66]:
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

In [67]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

log_reg2 = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
log_reg2.fit(X_train2, y_train2)

rf2 = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42, n_jobs=-1)
rf2.fit(X_train2, y_train2)

print("Both models trained successfully")

Both models trained successfully


In [68]:
log_reg2_scores = log_reg2.predict_proba(X_test2)[:, 1]
rf2_scores = rf2.predict_proba(X_test2)[:, 1]
y_test2_values = y_test2.values

print("Logistic Regression Precision@20:", precision_at_k(log_reg2_scores, y_test2_values, 20))
print("Random Forest Precision@20:", precision_at_k(rf2_scores, y_test2_values, 20))

Logistic Regression Precision@20: 0.5
Random Forest Precision@20: 0.35


In [69]:
log_reg2_top20_idx = np.argsort(-log_reg2_scores)[:20]
rf2_top20_idx = np.argsort(-rf2_scores)[:20]

overlap2 = set(log_reg2_top20_idx) & set(rf2_top20_idx)
print(f"Pages in both top-20 lists: {len(overlap2)} out of 20")

Pages in both top-20 lists: 0 out of 20


In [70]:
test_df2 = test_df2.copy()
test_df2["baseline_gap"] = test_df2["tier_median_ctr"] - test_df2["feat_ctr"]
test_df2["baseline_priority_score"] = test_df2["baseline_gap"] * test_df2["feat_impressions"]

baseline_precision_20 = precision_at_k(test_df2["baseline_priority_score"].values, y_test2_values, 20)
print("Baseline Precision@20 (on this test set):", baseline_precision_20)

Baseline Precision@20 (on this test set): 0.15


In [71]:
import pandas as pd

comparison_table = pd.DataFrame({
    "Method": ["Baseline (Week 4 rule)", "Logistic Regression", "Random Forest"],
    "Precision@20": [
        baseline_precision_20,
        precision_at_k(log_reg2_scores, y_test2_values, 20),
        precision_at_k(rf2_scores, y_test2_values, 20)
    ]
})

print(comparison_table)

                   Method  Precision@20
0  Baseline (Week 4 rule)          0.15
1     Logistic Regression          0.50
2           Random Forest          0.35


In [72]:
coef_df2 = pd.DataFrame({
    "feature": features_list,
    "coefficient": log_reg2.coef_[0]
}).sort_values("coefficient", key=abs, ascending=False)
print("Logistic Regression coefficients:")
print(coef_df2)

importance_df2 = pd.DataFrame({
    "feature": features_list,
    "importance": rf2.feature_importances_
}).sort_values("importance", ascending=False)
print("\nRandom Forest feature importances:")
print(importance_df2)

Logistic Regression coefficients:
             feature  coefficient
4           feat_ctr   -18.363975
1        feat_clicks    -0.707029
3   feat_days_active     0.506278
2  feat_avg_position    -0.012755
0   feat_impressions     0.001013

Random Forest feature importances:
             feature  importance
0   feat_impressions    0.414535
4           feat_ctr    0.221585
2  feat_avg_position    0.204319
3   feat_days_active    0.128803
1        feat_clicks    0.030758


**Label evolution:** My first attempt used `declining_flag` alone (impressions dropped
≥20% from first-half to second-half of March). Both models achieved Precision@20 = 0.45
against this label but I found the two models' top-20 lists had **zero overlap**,
despite identical precision, and their feature importances pointed in almost opposite
directions (Logistic Regression leaned heavily on `feat_ctr`; Random Forest leaned
heavily on `feat_avg_position` and `feat_impressions`). This suggested `declining_flag`
alone wasn't closely tied to my lane's actual question (CTR-vs-tier mismatch) — a page
can decline in impressions for reasons that have nothing to do with its title.

I revised to a **combined label**: `declining_flag AND ctr_mismatch`, where
`ctr_mismatch` is computed independently on this same first-half data (fresh tier
assignment, fresh tier medians — not reused from my Week 4 baseline numbers), requiring
CTR to be less than half the tier median AND at least 100 impressions (to avoid flagging
low-traffic noise as a "mismatch"). This tightened label is smaller (~10% positive, down
from ~33%) but much more directly tied to my lane's actual concern.

**Results on the combined label, same held-out test set (client-grouped split):**

| Method | Precision@20 |
|---|---|
| Baseline (Week 4 rule: `gap × impressions`) | 0.15 |
| Logistic Regression | 0.50 |
| Random Forest | 0.50 |

Both models substantially outperform the baseline on this metric. **Important caveat:**
my baseline was designed to rank purely by CTR-vs-tier mismatch — it has no way to
account for whether a page is also declining in impressions, since that signal isn't
part of its formula. The combined label requires both conditions, so part of the
baseline's lower score reflects being evaluated on a target it was never built to
predict, not purely a difference in modeling skill. A fairer like-for-like test would
also evaluate the baseline against `ctr_mismatch` alone.

**Model agreement:** Despite identical Precision@20, Logistic Regression and Random
Forest again shared **zero pages** in their top-20 lists. This is a recurring pattern
across both label versions I tried, and it suggests no single feature dominates cleanly
in this 5-feature set — different models can reach similar accuracy through very
different reasoning paths.

In [73]:
import pandas as pd

comparison_table = pd.DataFrame({
    "Method": ["Baseline (Week 4 rule)", "Logistic Regression", "Random Forest"],
    "Precision@20": [
        baseline_precision_20,
        precision_at_k(log_reg2_scores, y_test2_values, 20),
        precision_at_k(rf2_scores, y_test2_values, 20)
    ]
})

print(comparison_table)

                   Method  Precision@20
0  Baseline (Week 4 rule)          0.15
1     Logistic Regression          0.50
2           Random Forest          0.35


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*



**What the models lean on:**

Logistic Regression relies almost entirely on `feat_ctr` (coefficient -18.4, far larger
in magnitude than any other feature), while Random Forest spreads its attention more
evenly across `feat_impressions` (0.42), `feat_ctr` (0.22), and `feat_avg_position`
(0.20). Both models agree that CTR matters most which makes sense, since
`ctr_mismatch` (low CTR relative to tier) is one of the two conditions in my combined
label, and both models correctly picked up on the signal I deliberately built in.

**Where the models are wrong:**

I examined both models' top-20 picks that didn't actually match `combined_flag`. Both
Logistic Regression and Random Forest had 10 wrong picks out of 20, and every single
wrong pick for both models shares the same core failure: `ctr_mismatch = True`
but `declining_flag = False`. Neither model ever failed on the CTR-mismatch condition;
every miss came from the same source.

This tells a clear story: both models are genuinely good at detecting the CTR-mismatch
half of my label, but essentially guessing on the decline half. This makes sense for
two reasons. First, both models lean heavily on `feat_ctr` (dominant for Logistic
Regression, still second-most-important for Random Forest), so they're effectively
optimizing for CTR-mismatch detection rather than genuine decline prediction. Second,
and more fundamentally: none of my five features capture change over time at all
they're all snapshots computed from the first half of March only. The models have no
signal telling them whether a page's traffic is trending up or down heading into the
second half of the month, so they structurally cannot predict decline from static,
single-window features.

**A second, distinct error pattern in Random Forest specifically:** 7 of its 10 wrong
picks have `feat_ctr = 0.000000` exactly (zero clicks), and their impressions are much
lower than Logistic Regression's wrong picks (125–1,340, versus 4,000–25,000 for
Logistic Regression). Since `feat_impressions` is Random Forest's single most important
feature, a page with any real traffic and zero clicks likely looks "significant" to the
model, even when the actual traffic volume is small. This suggests Random Forest is
getting fooled by small, low-traffic, zero-click pages flagging them as high-priority
even though their low absolute impression count means the CTR problem, if real, affects
relatively few searchers. Logistic Regression's stronger reliance on the CTR value
itself (rather than raw impressions) seems to make it less susceptible to this specific
error.

**What this means for the model vs. baseline comparison:**

Both models beat the baseline (0.50 vs. 0.15 Precision@20), but this error analysis
shows *why*, honestly: the win comes almost entirely from being much better at CTR-
mismatch detection than the baseline's simple `gap × impressions` formula, not from
successfully predicting decline. A model that added real time-trend features (e.g.,
week-over-week impression change *within* the first half, rather than only totals)
would likely be needed to meaningfully improve on the decline-prediction half of
this label.

In [74]:
test_df2["log_reg2_score"] = log_reg2_scores
test_df2["rf2_score"] = rf2_scores

# Which pages did Logistic Regression's top 20 get WRONG?
log_reg2_top20 = test_df2.iloc[np.argsort(-log_reg2_scores)[:20]]
log_reg2_wrong = log_reg2_top20[~log_reg2_top20["combined_flag"]]

print(f"Logistic Regression top-20: {len(log_reg2_wrong)} wrong out of 20")
print(log_reg2_wrong[["content_hash_id", "feat_ctr", "feat_impressions", "feat_avg_position", "declining_flag", "ctr_mismatch"]])

Logistic Regression top-20: 10 wrong out of 20
                 content_hash_id  feat_ctr  feat_impressions  \
87340   content_39457d17e716086c  0.000280           25042.0   
12070   content_9a4594adab0f2c81  0.000734           14995.0   
11760   content_567d370cf1fdbd1d  0.000649           12318.0   
43397   content_1bc14f25ea5989e0  0.000369            8134.0   
88078   content_d1db17521a55d9fc  0.000997           19059.0   
87488   content_d7982b8ad5e27e6e  0.000326            6128.0   
87254   content_4977e90c4d93cf9f  0.001039           15397.0   
11780   content_1c38733f5292a291  0.000000            4373.0   
11444   content_3cb586f697000543  0.000209            4783.0   
107366  content_31ea2c45cdc6c46e  0.000241            4152.0   

        feat_avg_position  declining_flag  ctr_mismatch  
87340           38.271078           False          True  
12070           34.689023           False          True  
11760           38.718105           False          True  
43397           

In [75]:
# Which pages did Random Forest's top 20 get WRONG?
rf2_top20 = test_df2.iloc[np.argsort(-rf2_scores)[:20]]
rf2_wrong = rf2_top20[~rf2_top20["combined_flag"]]

print(f"Random Forest top-20: {len(rf2_wrong)} wrong out of 20")
print(rf2_wrong[["content_hash_id", "feat_ctr", "feat_impressions", "feat_avg_position", "declining_flag", "ctr_mismatch"]])

Random Forest top-20: 13 wrong out of 20
                 content_hash_id  feat_ctr  feat_impressions  \
11446   content_53554cda18c15a47  0.000746            1340.0   
80136   content_73f4ee7d7248b850  0.000000             302.0   
119397  content_144e6c7eb5851a16  0.000000             209.0   
80128   content_d519417d47e3973d  0.000000             201.0   
119330  content_7469d66d333e1403  0.000000             738.0   
11807   content_aace0d1032d3fd7b  0.000000             125.0   
11364   content_f3f7f1f0ca34e560  0.000000             331.0   
31520   content_ee529f735a3b05cf  0.000331            3021.0   
86535   content_a4358af024086286  0.000000             248.0   
12586   content_c253dd6f4296eeb9  0.001835             545.0   
87443   content_5afb127d37720439  0.000000             108.0   
11423   content_f6b812646ebf6de2  0.000000             357.0   
31687   content_e75974edbd90fc60  0.000000             408.0   

        feat_avg_position  declining_flag  ctr_mismatch  
1144

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.